In [3]:
import time
import pickle
import json
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium_stealth import stealth

def setup_driver():
    """Настройка драйвера с обходом обнаружения автоматизации."""
    chrome_options = Options()
    chrome_options.page_load_strategy = 'eager'
    chrome_options.binary_location = "/usr/bin/chromium-browser"  # путь к Chromium (может отличаться)
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option("useAutomationExtension", False)
    
    service = Service("/usr/bin/chromedriver")  # путь к chromedriver (может отличаться)
    driver = webdriver.Chrome(service=service, options=chrome_options)
    
    # Скрываем следы автоматизации
    stealth(driver,
            languages=["ru-RU", "ru"],
            vendor="Google Inc.",
            platform="Win32",
            webgl_vendor="Intel Inc.",
            renderer="Intel Iris OpenGL Engine",
            fix_hairline=True)
    
    return driver

def login(driver, email, password):
    """Выполняет вход в mail.tm."""
    
    # print(2)
    driver.get("https://mail.tm/")
    wait = WebDriverWait(driver, 20)
    # print("wait")
    
    # Закрываем возможное куки-уведомление
    # try:
    #     cookie_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "button[aria-label='Close']")))
    #     cookie_btn.click()
    # except:
    #     pass
    
    menue_btn = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'button[id="reka-dropdown-menu-trigger-v-1-4"]'))) 
    # print("menue_btn")
    menue_btn.click()

    cl = "group relative w-full flex items-center select-none outline-none before:absolute before:z-[-1] before:inset-px before:rounded-md data-disabled:cursor-not-allowed data-disabled:opacity-75 data-[state=open]:text-highlighted transition-colors before:transition-colors p-1.5 text-sm gap-1.5 text-secondary data-highlighted:text-secondary data-highlighted:before:bg-secondary/10 data-[state=open]:before:bg-secondary/10"
    enter_button = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, f'button[class="{cl}"]'))) 
    # print("enter_button")
    enter_button.click()

    # Поле email
    email_input = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "input[name='address']")))
    # print("email_input")
    email_input.clear()
    email_input.send_keys(email)
    
    # Поле пароля
    password_input = driver.find_element(By.CSS_SELECTOR, "input[name='password']")
    # print("password_input")
    password_input.clear()
    password_input.send_keys(password)
    
    # Кнопка входа
    cl = "w-full inline-flex justify-center border border-transparent rounded-md bg-indigo-600 px-4 py-2 text-base font-medium leading-6 text-white shadow-sm transition focus:border-indigo-700 dark:bg-indigo-700 hover:bg-indigo-500 sm:text-sm sm:leading-5 focus:outline-none focus:ring-indigo dark:hover:bg-indigo-800"
    login_btn = driver.find_element(By.CSS_SELECTOR, f'button[class="{cl}"]')
    # print("login_btn")
    login_btn.click()
    
    # Ждём перехода на страницу почты (появление списка писем)
    cl = "group block transition hover:bg-gray-50 focus:outline-none dark:focus:bg-gray-900/50 dark:hover:bg-gray-900/50"
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, f'a[class="{cl}"]')))
    print("Вход выполнен успешно.")

def get_inbox_emails(driver, limit=10):
    """
    Возвращает список элементов писем (каждый элемент — DOM-узел письма).
    Ожидает загрузки списка и ограничивает количество.
    """
    wait = WebDriverWait(driver, 15)
    # Список писем — внутри .message-list, каждое письмо — .message
    cl = "flex items-center px-4 py-4 sm:px-6"
    messages = wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, f'div[class="{cl}"]')))
    # print("Messages found")
    return messages[:limit]

def read_email(driver, email_element):
    """
    Кликает по письму, извлекает тему, отправителя, дату, тело.
    Возвращает словарь с данными.
    """

    # Кликаем по письму
    email_element.click()
    wait = WebDriverWait(driver, 10)
    # print("email_element")

    # Ждём, пока откроется область просмотра письма
    cl = "mt-4 overflow-hidden border-b border-gray-200 bg-white px-4 py-5 shadow dark:border-gray-900 sm:rounded-md sm:px-6"
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, f'div[class="{cl}"]')))
    # print("wait")

    # Извлекаем данные
    try:
        cl = "text-2xl font-bold leading-7 text-gray-900 sm:truncate dark:text-gray-300"
        subject = driver.find_element(By.CSS_SELECTOR, f'h2[class="{cl}"]').text
        # print("subject")
    except:
        subject = "Нет темы"
    
    try:
        cl = "text-sm font-normal leading-5 text-gray-700"
        sender = driver.find_element(By.CSS_SELECTOR, f'span[class="{cl}"]').text
        # print("sender")
    except:
        sender = "Неизвестный отправитель"
    
    try:
        cl = "truncate text-gray-500"
        date = driver.find_element(By.CSS_SELECTOR, f'time[class="{cl}"]').get_attribute('datetime')
        # print("date")
    except:
        date = "Дата не указана"
    

    try:
        iframe = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "#iFrameResizer0")))
        
        print("Owwo")
        print(driver.find_element(By.CSS_SELECTOR, "#iFrameResizer0").text)
        print(driver.find_element(By.CSS_SELECTOR, "#iFrameResizer0").get_attribute("innerHTML"))
        print(driver.find_element(By.CSS_SELECTOR, "#iFrameResizer0").get_attribute("outerHTML"))
        print(driver.find_element(By.CSS_SELECTOR, "#iFrameResizer0").get_attribute("innerText"))
        print(driver.find_element(By.CSS_SELECTOR, "#iFrameResizer0").get_attribute("textContent"))
        print("Owwo")
        body_text = iframe.find_element(By.CSS_SELECTOR, "#iFrameResizer0").text
    except:
        body_text = ""
    # body_text = ""

    # try:
    #     # Явно ждём появления iframe с id iFrameResizer0
    #     iframe = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "iframe#iFrameResizer0")))
    #     srcdoc = iframe.get_attribute("srcdoc")
    #     print(f"Длина srcdoc: {len(srcdoc)}")
    #     if srcdoc:
    #         from bs4 import BeautifulSoup
    #         soup = BeautifulSoup(srcdoc, 'html.parser')
    #         body_text = soup.body.get_text(separator="\n") if soup.body else ""
    # except Exception as e:
    #     print(f"Ошибка при парсинге srcdoc: {e}")

    
    # Закрываем просмотр письма (кнопка "назад" или просто возвращаемся в список)
    # На mail.tm кнопка возврата имеет aria-label="Back"
    try:
        cl = "flex items-center text-base font-medium leading-5 text-gray-500 transition dark:text-gray-400 hover:text-gray-700 dark:hover:text-gray-500"
        back_btn = driver.find_element(By.CSS_SELECTOR, f'a[class="{cl}"]')
        # print("back_btn")
        back_btn.click()
    except:
        # Если кнопки нет, можно обновить список, но проще вернуться в историю
        driver.back()
    
    # Ждём возврата к списку писем
    
    cl = "flex items-center px-4 py-4 sm:px-6"
    WebDriverWait(driver, 10).until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, f'div[class="{cl}"]')))
    
    return {
        "subject": subject,
        "sender": sender,
        "date": date,
        "body": body_text
    }



In [4]:
with open('mail.json', 'r', encoding='utf-8') as file:
    login_dict = json.load(file)

EMAIL = login_dict["EMAIL"]
PASSWORD = login_dict["PASSWORD"]

    
driver = setup_driver()
try:
    # print(1)
    login(driver, EMAIL, PASSWORD)
    
    # Получаем последние 5 писем
    emails = get_inbox_emails(driver, limit=5)
    print(f"Найдено писем: {len(emails)}")
    
    # Проходим по каждому письму и выводим информацию
    limit = 5
    for i in range(limit):
        # Получаем свежий список писем на каждой итерации
        emails = get_inbox_emails(driver, limit=limit)
        if i >= len(emails):
            print(f"Получено только {len(emails)} писем")
            break
        email_elem = emails[i]
        print(f"\n--- Письмо {i+1} ---")
        data = read_email(driver, email_elem)
        print(f"От: {data['sender']}")
        print(f"Тема: {data['subject']}")
        print(f"Дата: {data['date']}")
        print(f"Текст: {data['body'][:200]}...")
    
finally:
    driver.quit()


Вход выполнен успешно.
Найдено писем: 2

--- Письмо 1 ---
Owwo


<iframe data-v-cdf33a98="" class="w-full" id="iFrameResizer0" scrolling="yes" style="overflow: auto;"></iframe>


Owwo
От: leousovs@gmail.com
Тема: w354
Дата: 2026-03-25T16:14:26+00:00
Текст: ...

--- Письмо 2 ---
От: leousovs@gmail.com
Тема: HHi
Дата: 2026-03-24T21:35:28+00:00
Текст: ...
Получено только 2 писем
